# MCP(Model Context Protocol)

### MCP(Model Context Protocol) 정의

**MCP**는 **AI 애플리케이션과 외부 도구 간의 상호작용을 표준화**한 오픈 프로토콜입니다.  
즉, LLM이 다양한 외부 시스템(API, 데이터베이스, 도구 등)과 일관된 방식으로 통신할 수 있도록 하는 **표준 인터페이스 계층**입니다.

---

### MCP의 4가지 핵심 특징

| 특징 | 설명 | 장점 | 예시 |
|------|------|------|------|
| **표준화된 도구 인터페이스** | 모든 외부 도구가 동일한 프로토콜로 연결 | 새로운 도구 추가 시 별도 학습 불필요 | 날씨 API, DB, 파일 시스템 등 동일 방식 호출 |
| **다양한 전송 메커니즘** | **공식: `stdio`, `Streamable HTTP` (구 `HTTP+SSE`는 deprecated)** | 환경(로컬/원격)에 맞는 최적 방식 선택 | 로컬은 `stdio`, 원격은 `Streamable HTTP` |
| **동적 도구 검색** | 런타임 시 사용 가능한 도구 자동 탐색 | 코드 수정 없이 확장 가능 | 새로운 플러그인 자동 인식 |
| **확장 가능한 아키텍처** | 모듈형 구조 및 멀티 서버 지원 | 시스템 규모 확장에 유연 | 필요 기능만 선택적 추가 |

---

### MCP 전송 메커니즘 비교

| 구분 | **stdio** | **HTTP+SSE (레거시)** | **Streamable HTTP (현행)** |
|------|------------|------------------------|-----------------------------|
| **전송 계층** | 표준입출력 파이프(로컬 IPC) | GET(SSE 수신) + POST(송신) 2 엔드포인트 | 단일 HTTP 엔드포인트(POST/GET), **SSE로 응답 스트리밍** |
| **통신 구조** | 한 프로세스 내 파이프 기반 | S→C는 SSE, C→S는 POST로 분리 | C→S: POST / S→C: **SSE 스트림** → 실질적 양방향 통신 |
| **실시간성 / 재개 지원** | 빠름(로컬 IPC 수준), 재개 불가 | 단방향 스트림만 가능, 재개 미지원 | **Event ID / `Last-Event-ID` 기반 스트림 재개 가능** |
| **적합 환경** | 로컬 도구·IDE 플러그인 | 과거 원격 MCP 서버 호환 | **최신 원격 MCP 서버 / 실시간 스트리밍 서비스** |
| **표준 지위** | 지속 지원 | **Deprecated (2025-03-26)** | **Current Standard (Streamable HTTP)** |

> ⚙️ **보안 주의사항:**  
> - Streamable HTTP 사용 시 **Origin 검증 및 로컬(127.0.0.1) 바인딩** 권장  
> - 외부 네트워크 통신 시 **토큰 기반 인증(OAuth 등)** 필수  
> - HTTP+SSE는 호환성 목적 외에는 더 이상 권장되지 않음

---

### 요약

- **stdio** → 로컬 환경용, 빠르고 안정적  
- **Streamable HTTP** → 현대적 원격 통신 표준, SSE 기반 실시간 양방향 스트리밍 지원  
- **HTTP+SSE** → 과거 임시 방식으로 사용되었으나 **2025년 3월 이후 폐기됨**

---

### 핵심 포인트

> 🔎 **MCP는 "AI와 도구를 연결하는 표준 인터페이스"**  
> - 도구 검색, 요청·응답 포맷, 스트리밍, 인증·보안 정책을 통합적으로 관리  
> - 로컬(예: IDE 확장)에는 `stdio`,  
>   원격(예: 클라우드 도구 서버)에는 `Streamable HTTP`를 사용하는 것이 권장됨.


In [1]:
# 비동기 처리를 위한 nest_asyncio (Jupyter 환경에서 필수)
import nest_asyncio
from typing import List, Dict, Any

# OpenAI LLM 모델 사용을 위한 import
from langchain_openai import ChatOpenAI

# LangGraph의 사전 구축된 React 에이전트 import
from langgraph.prebuilt import create_react_agent

# 메모리 기반 체크포인터 (대화 상태 저장)
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트 - 여러 MCP 서버를 동시에 관리하는 핵심 컴포넌트
from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API KEY 정보로드
load_dotenv(override=True)

# 비동기 호출 활성화
nest_asyncio.apply()

In [3]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("LangChain-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangChain-Tutorial


## MCP 다양한 서버 제작 실습

### 서버 파일 구성 및 특징

이 튜토리얼에서는 세 가지 타입의 **MCP 서버**를 사용합니다.  
각 서버는 전송 방식과 역할이 다르며, `MultiServerMCPClient`로 통합 관리할 수 있습니다.

| 서버 유형 | 파일명 | 전송 방식 | 주요 기능 | 사용 시나리오 |
|-----------|---------|------------|------------|----------------|
| **로컬 서버** | `mcp_server_local.py` | stdio | 파일 관리, 로컬 데이터 조회 | 빠른 응답이 필요한 기본 기능 |
| **원격 서버** | `mcp_server_remote.py` | Streamable HTTP | 시간 조회, 외부 API 연동 | 네트워크를 통한 원격 서비스 |
| **검색 서버** | `mcp_server_rag.py` | stdio | RAG 문서 검색, 벡터 유사도 검색 | 대용량 문서에서 정확한 정보 검색 |

> 💡 **참고:** MCP 공식 스펙 기준으로 `WebSocket`은 지원 예시로 언급될 수 있지만,  
> 실제 표준 전송 방식은 `stdio`와 `Streamable HTTP` 두 가지입니다.  
> (`HTTP+SSE`는 2025년 3월부로 폐기됨)

---

### 서버별 상세 특징

#### 🖥️ 로컬 서버 (`stdio`)
- **장점**  
  - 빠른 응답속도  
  - 안정적 연결 (네트워크 불필요)  
  - OS 파일 접근이 용이  
- **단점**  
  - 동일 머신 내에서만 실행 가능  
- **적용 분야**  
  - 로컬 파일 시스템 제어, 오프라인 데이터 처리, IDE 통합 도구

---

#### ☁️ 원격 서버 (`Streamable HTTP`)
- **장점**  
  - 네트워크를 통한 원격 호출 가능  
  - 확장성과 보안 설정 용이 (토큰 인증, Origin 검증 등)  
  - SSE 기반 **스트리밍 응답 및 재개 지원**  
- **단점**  
  - 네트워크 지연 및 연결 관리 필요  
- **적용 분야**  
  - 외부 API 연동, 클라우드 서비스 연결, 실시간 스트리밍 응답 제공

---

#### 🔍 검색 서버 (`stdio + RAG`)
- **장점**  
  - 대용량 문서 처리 및 의미 기반 검색 지원  
  - 벡터 유사도 검색으로 정교한 컨텍스트 제공  
- **단점**  
  - 초기 임베딩 구축 및 리소스 사용량 높음  
- **적용 분야**  
  - 사내 지식베이스 검색, 문서 QA 시스템, LLM 보조 검색 엔진

---

### 서버 분리의 장점

| 장점 | 설명 | 실제 적용 예시 |
|------|------|----------------|
| **모듈성** | 각 기능을 독립적으로 개발·배포 가능 | 날씨 서버 오류 시에도 시간 서버는 정상 작동 |
| **확장성** | 필요한 기능만 선택적으로 추가 가능 | 번역 서버 추가 시 기존 서버 영향 없음 |
| **안정성** | 장애가 다른 서버로 전이되지 않음 | 검색 서버 장애 시에도 나머지 서비스 유지 |
| **성능 최적화** | 서버별 리소스 설정 조정 가능 | 검색 서버는 GPU, 시간 서버는 CPU 경량 운영 |

## MultiServerMCPClient

**`MultiServerMCPClient`** 는 **여러 MCP 서버를 하나의 통합 인터페이스로 관리**하는 핵심 컴포넌트입니다.  
각 서버(`stdio`, `Streamable HTTP`)에서 제공하는 도구들을 자동으로 수집하고,  
단일 클라이언트에서 일관된 방식으로 호출할 수 있도록 설계되어 있습니다.

---

### MultiServerMCPClient의 핵심 기능

| 기능 | 설명 | 코드 예시 | 장점 |
|------|------|-----------|------|
| **서버 구성 관리** | 각 MCP 서버 설정을 딕셔너리로 정의·관리 | `server_configs = {"weather": {...}}` | 설정 변경 및 버전 관리 용이 |
| **동적 도구 로딩** | 각 서버가 제공하는 도구를 자동 탐색·통합 | `tools = await client.get_tools()` | 코드 수정 없이 실시간 도구 업데이트 |
| **통합 인터페이스** | 모든 서버의 도구를 하나의 통합 리스트로 제공 | `for tool in tools: result = await tool.invoke()` | 서버별 호출 방식 차이 제거 |
| **에러 처리 및 복구** | 특정 서버 장애 시 다른 서버는 정상 작동 | 내부 재연결 및 로깅 처리 | 시스템 안정성 향상 |

---

### 서버 구성 설정 상세

#### 기본 구성 형태
```python
server_configs = {
    "서버명": {
        "command": "실행명령어",      # uv, python, node 등
        "args": ["인자1", "인자2"],   # 실행 시 전달할 인자
        "transport": "전송방식",      # stdio 또는 streamable_http
    }
}
```

📘 stdio 전송 방식 예시
```python
"local_server": {
    "command": "uv",
    "args": ["run", "python", "server/mcp_server_local.py"],
    "transport": "stdio",
}
```

🌐 Streamable HTTP 전송 방식 예시
```python
"remote_server": {
    "url": "http://127.0.0.1:8002/mcp",
    "transport": "streamable_http",
}
```

In [4]:
# MultiServerMCPClient 설정 예제
async def setup_mcp_client(server_configs: List[Dict[str, Any]]):
    """
    MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 설정 정보 리스트

    Returns:
        tuple: (MCP 클라이언트, 도구 리스트)
    """

    # MultiServerMCPClient 인스턴스 생성 - 여러 MCP 서버를 통합 관리
    client = MultiServerMCPClient(server_configs)

    # 모든 연결된 서버로부터 사용 가능한 도구들을 수집
    tools = await client.get_tools()

    # 로드된 도구 정보를 콘솔에 출력 (디버깅 및 확인용)
    print(f"✅ {len(tools)} 개의 MCP 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")  # 각 도구의 이름 출력

    return client, tools

### 로컬 MCP 서버 방식 사용

In [7]:
# 서버 구성 정의 - 로컬 날씨 서버 설정
server_configs = {
    "weather": {
        "command": "python",  # uv → python으로 변경
        "args": [
            "02-MCP/server/mcp_server_local.py",
        ],  # 날씨 서버 스크립트 실행
        "transport": "stdio",  # 표준 입출력 방식으로 통신
    },
}

# MCP 클라이언트 생성 및 도구 로딩
client, tools = await setup_mcp_client(server_configs=server_configs)

  + Exception Group Traceback (most recent call last):
  |   File "/Users/oksjjj/coding/mysuni_aiagent/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_85004/1932973513.py", line 13, in <module>
  |     client, tools = await setup_mcp_client(server_configs=server_configs)
  |                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_85004/4043936694.py", line 17, in setup_mcp_client
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/oksjjj/coding/mysuni_aiagent/.venv/lib/python3.11/site-packages/langchain_mcp_adapters/client.py", line 142, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^